# Dataset Preparation for MR-LFADS

This tutorial explains how to format your dataset to be compatible with the standard data structure for the MR-LFADS datamodule, `BasicDataModule`.

## 1. Data Structure

The `mrlfads.datamodules.BasicDataModule` is the standard datamodule for MR-LFADS. It expects an HDF5 file (`data.h5`) with the following structure:

```text
data.h5
├── session_index/
│   ├── area-<brain_area_name>      # neural activity (type: "hidden_state")
│   ├── inputs-<brain_area_name>    # inputs to a brain area (type: "inputs")
│   └── info                        # metadata (type: "info")
```

Each `session_index` group corresponds to a single recording session.

## 2. Dataset Requirements

Within each `session_index`, datasets must follow these conventions:

### (1) Neural Activity
- **Naming**: area-`<brain_area_name>`
- **Type**: `"hidden_state"`
- **Shape**: `(n_trials, n_time, n_neurons)`
- **Description**: Neural activity for a specific brain area `<brain_area_name>`
    
### (2) Inputs (optional)
- **Naming**: inputs-`<brain_area_name>`
- **Type**: `"inputs"`
- **Shape**: `(n_trials, n_time, n_input_features)`
- **Description**: External inputs associated with a specific brain area `<brain_area_name>`

### (3) Metadata (optional)
- **Naming**: arbitrary 
- **Type**: `"info"`  
- **Shape**: `(n_trials, n_time, n_info_dims)`  
- **Description**: Session-level metadata aligned to each trial and time step  

## 3. Test Your Dataset

To verify that your dataset loads correctly, specify the data file and brain area names:

In [ ]:
filename = NotImplemented  # path relative to config.paths.datapath
area_names = NotImplemented  # list of area names, e.g. ["Area1", "Area2"]

Then initialize and setup the datamodule:

In [ ]:
from mrlfads.datamodules import BasicDataModule

dm = BasicDataModule(
    filename=filename,
    area_names=area_names,
    session_idxs=[0],
)
dm.setup()

### Notes

* `filename` should point to your data.h5 file relative to config.paths.datapath
* `area_names` must match the suffixes used in your dataset (e.g., area-M1 → "M1")
* `session_idxs=[0]` loads the first session; adjust if multiple sessions are present

## 4. Under the Hood (Optional)

For a deeper understanding of how data is passed into the `mrlfads.model.MRLFADS` model, this section describes the internal structure.

### Batch Structure

For each `session_index`, data is passed to the model in the following format:

```text
session_idx
├── Main data (type: `mrlfads.utils.common_utils.Batch`)
│   ├── neural data
│   └── external inputs
└── Metadata (type: dict)
    ├── <metadata_name_1>
    └── ...
```

* Main data contains neural activity and inputs, and is stored as `model.current_batch`
* Metadata contains the trial-aligned metadata, and is stored as ``model.current_info`
        
### Notes 

**Important Note** on `session_idx`: The session_idx used here refers to the index within the selected sessions, not the original dataset index. For example:

If your datamodule is initialized with session_idxs = [0, 4, 5], then batches will use:

`session_idx ∈ {0, 1, 2}`

corresponding to dataset sessions [0, 4, 5], respectively.

### Try It Yourself

If you have a stored MRLFADS model, you can inspect this structure directly using the following code:

In [ ]:
import os

checkpoint_path = NotImplemented  # absolute path of stored MRLFADS model
config_path = os.path.join(checkpoint_path, 'configs', 'main.yaml')

In [ ]:
from mrlfads.run import load

# Load the model and run one forward pass on the validation dataset
state_dict = load(
    config_path=config_path,
    validate=True,
)

# Extract the loaded model
model = state_dict['model']

In [ ]:
# Take one example session, one example area
session_idx = 0
area_name = model.area_names[0]

print(f'The shape of neural activity for area {area_name}, session {session_idx} is: ')
print(model.current_batch[session_idx].encod_data[area_name].shape)

In [ ]:
print(f'The metadata keys are:')
print(model.current_info[session_idx].keys())